In [13]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List

load_dotenv()

True

In [14]:
# Setup
persistent_directory = "dbv2/chroma_db"
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [15]:
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

In [16]:
# Pydantic model for structured output
class QueryVariations(BaseModel):
    queries: List[str]

In [17]:
# Original query
original_query = "what is Guinea-worm?"
print(f"Original Query: {original_query}\n")


Original Query: what is Guinea-worm?



# # Step 1: Generate Multiple Query Variations

In [18]:
llm_tools = llm.with_structured_output(QueryVariations)

prompt = f"""Generate 3 different variations of this query that would help retrieve relevant documents:

Original query: {original_query}

Return 3 alternative queries that rephrase or approach the same question from different angles."""


result = llm_tools.invoke(prompt)
Query_Variations = result.queries

print("Generated Query Variations:")
for i, variation in enumerate(Query_Variations, 1):
    print(f"{i}. {variation}")

Generated Query Variations:
1. Can you explain what Guinea-worm disease is?
2. What are the causes and symptoms of Guinea-worm infection?
3. How does the Guinea-worm parasite affect humans?


#  Step 2: Search with Each Query Variation & Store Results

In [19]:
retriever = db.as_retriever(search_kwargs={"k": 5})  # Get more docs for better RRF
all_retrieval_results = []  # Store all results for RRF

for i, query in enumerate(Query_Variations, 1):
    print(f"\n=== RESULTS FOR QUERY {i}: {query} ===")
    
    docs = retriever.invoke(query)
    all_retrieval_results.append(docs)  # Store for RRF calculation
    
    print(f"Retrieved {len(docs)} documents:\n")
    
    for j, doc in enumerate(docs, 1):
        print(f"Document {j}:")
        print(f"{doc.page_content[:150]}...\n")


=== RESULTS FOR QUERY 1: Can you explain what Guinea-worm disease is? ===
Retrieved 5 documents:

Document 1:
2.4.2.4 Guinea-worm

In this infection the pathogen, a large worm, creates a blister on the person’s skin, which erupts when it comes into contact wit...

Document 2:
2.4.2.3 Beef/pig tapeworm infection

The pathogens leave the person through faeces. The excreted eggs then have to be ingested by either cattle or pig...

Document 3:
2.4.2.1 Soil-transmitted helminths

These worms leave the body through faeces as eggs or larvae. After excretion they have to develop in soil. They ca...

Document 4:
2.4.2 Infections with indirect transmission

A pathogen with an indirect transmission route must go through a development phase outside the host befor...

Document 5:
DISEASE AND DISEASE TRANSMISSION

Chapter 2

Disease and disease transmission

An enormous variety of organisms exist, including some which can surviv...


=== RESULTS FOR QUERY 2: What are the causes and symptoms of Guin